In [7]:
import requests
import pandas as pd 
import numpy as np 
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time 
import calendar

In [9]:
from datetime import datetime
now=datetime.now()
now.strftime("%Y-%m-%dT%H:%M:%S.%f")

'2026-03-25T13:59:21.651143'

In [249]:
data=[]
api_key='e62c395ec9844095ada4b688d012a975'
technical='time_series'
ticker='BTC/USD'
timezone='Asia/Calcutta'
interval='30min'
for yr in range(2022,2026):
    for m in range(1,5): 
        last_day=calendar.monthrange(yr,3*m)[1]   
        start_date=f'{yr}-{(3*m-2):02d}-01T00:00:00'
        end_date=f'{yr}-{(3*m):02d}-{last_day}T23:30:00'
        api_url=f'https://api.twelvedata.com/{technical}?symbol={ticker}&order=asc&timezone={timezone}&start_date={start_date}&end_date={end_date}&interval={interval}&outputsize=5000&apikey={api_key}'
        while True:
            session=requests.Session()
            retries=Retry(total=5,backoff_factor=1,status_forcelist=[429,500,502,503,504])
            session.mount("https://",HTTPAdapter(max_retries=retries))
            fetch=session.get(api_url).json()
            if "values" in fetch:
                df=pd.DataFrame(fetch['values'])
                data.append(df)
                time.sleep(8)
                break
            elif "code" in fetch and fetch["code"]==429:
                print("Rate limit exceeded")
                time.sleep(15)
            else: 
                print("API Error:",fetch)
                fetch=None
                break
data=pd.concat(data,ignore_index=True)
data['color']=np.where(data['close']>data['open'],1,0)
data['open']=pd.to_numeric(data['open'])
data['close']=pd.to_numeric(data['close'])
data['abs_body']=abs(data['close']-data['open'])
data['body']=(data['close']-data['open'])
data['high']=pd.to_numeric(data['high'])
data['low']=pd.to_numeric(data['low'])
data['upper_wick']=np.where(data['color']==1,data['high']-data['close'],data['high']-data['open'])
data['lower_wick']=np.where(data['color']==1,data['open']-data['low'],data['close']-data['low'])
data=data.dropna()
data=data.reset_index(drop=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075.00000,46080.87109,0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,1,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
69992,2025-12-31 21:30:00,87830.030000,87932.17000,87558.00000,87792.55000,0,37.480000,-37.480000,102.140000,234.55000
69993,2025-12-31 22:00:00,87797.910000,87813.02000,87259.01000,87531.06000,0,266.850000,-266.850000,15.110000,272.05000
69994,2025-12-31 22:30:00,87531.060000,87637.99000,87356.29000,87496.24000,0,34.820000,-34.820000,106.930000,139.95000
69995,2025-12-31 23:00:00,87496.250000,87550.34000,87400.76000,87487.96000,0,8.290000,-8.290000,54.090000,87.20000


In [250]:
data['datetime']=pd.to_datetime(data['datetime'])
missing_times=pd.date_range(start="2022-01-01 00:00",end="2025-12-31 23:30",freq="30min").difference(data['datetime'])
missing_times

DatetimeIndex(['2022-02-19 19:30:00', '2022-02-19 20:00:00',
               '2022-02-19 20:30:00', '2022-02-19 21:00:00',
               '2022-02-19 21:30:00', '2022-02-19 22:00:00',
               '2022-02-19 22:30:00', '2022-02-19 23:00:00',
               '2022-02-19 23:30:00', '2022-02-20 00:00:00',
               ...
               '2025-10-25 21:30:00', '2025-10-25 22:00:00',
               '2025-10-25 22:30:00', '2025-10-25 23:00:00',
               '2025-10-25 23:30:00', '2025-10-26 00:00:00',
               '2025-10-26 00:30:00', '2025-10-26 01:00:00',
               '2025-10-26 01:30:00', '2025-10-26 02:00:00'],
              dtype='datetime64[ns]', length=131, freq=None)

In [6]:
missing_times=pd.date_range(start="2022-02-19 19:30",end="2022-02-20 02:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[40417.019531, 40382.390620, 40495.820310, 40316.121090, 39856.980470, 40183.230470, 39976.660160, 40073.031250, 40052.289060, 39966.011719, 40165.449220, 40245.160160, 40225.781250, 40079.878910, 39969.539060],
    "high":[40497.32812, 40449.48047, 40739.82031, 40316.12109, 40377.17188, 40382.75, 40360.03125, 40164.83984, 40052.28906, 40194.71875, 40268.60156, 40353.83984, 40294.37891, 40178.96094, 40149.51172],
    "low":[40316.82031, 39673.44922, 40278.39844, 39693.89844, 39464.60938, 40004.41016, 39976.66016, 39963.35156, 39825.62109, 39943.26172, 40084.64062, 40094.92969, 40050.39062, 39960.55859, 39916.57031],
    "close":[40354.148440, 40437.609380, 40278.398440, 39790.531250, 40153.898440, 40004.410160, 40095.351560, 40032.710940, 39983.800780, 40193.539060, 40268.601560, 40218.441410, 40123.281250, 39966.988280, 40128.75],
    "abs_body":[62.871091,55.21876,217.42187,525.58984,296.91797,178.82031,118.6914,40.32031,68.48828,227.527341,103.15234,26.71875,102.5,112.89063,159.21094],
    "body":[-62.871091,55.21876,-217.42187,-525.58984,296.91797,-178.82031,118.6914,-40.32031,-68.48828,227.527341,103.15234,-26.71875,-102.5,-112.89063,159.21094]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

NameError: name 'pd' is not defined

In [271]:
missing_times=pd.date_range(start="2022-04-03 02:00",end="2022-04-03 02:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[46324.46094],
    "high":[46391.42188],
    "low":[46111.011719],
    "close":[46164.92969],
    "abs_body":[159.53125],
    "body":[-159.53125]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.179690,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075.000000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.550780,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.429690,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.949220,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70008,2022-02-20 01:00:00,40245.160160,40353.83984,40094.929690,40218.44141,NaN,26.718750,-26.718750,NaN,NaN
70009,2022-02-20 01:30:00,40225.781250,40294.37891,40050.390620,40123.28125,NaN,102.500000,-102.500000,NaN,NaN
70010,2022-02-20 02:00:00,40079.878910,40178.96094,39960.558590,39966.98828,NaN,112.890630,-112.890630,NaN,NaN
70011,2022-02-20 02:30:00,39969.539060,40149.51172,39916.570310,40128.75000,NaN,159.210940,159.210940,NaN,NaN


In [272]:
missing_times=pd.date_range(start="2022-05-06 00:30",end="2022-05-06 01:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[39601.14844,39844.64062],
    "high":[39972.55078,39903.19922],
    "low":[39545.058594,39671.89844],
    "close":[39840.53906,39816.12109],
    "abs_body":[239.39062,28.51953],
    "body":[239.39062,-28.51953]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.179690,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075.000000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.550780,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.429690,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.949220,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70010,2022-02-20 02:00:00,40079.878910,40178.96094,39960.558590,39966.98828,NaN,112.890630,-112.890630,NaN,NaN
70011,2022-02-20 02:30:00,39969.539060,40149.51172,39916.570310,40128.75000,NaN,159.210940,159.210940,NaN,NaN
70012,2022-04-03 02:00:00,46324.460940,46391.42188,46111.011719,46164.92969,NaN,159.531250,-159.531250,NaN,NaN
70013,2022-05-06 00:30:00,39601.148440,39972.55078,39545.058594,39840.53906,NaN,239.390620,239.390620,NaN,NaN


In [273]:
missing_times=pd.date_range(start="2022-10-22 17:30",end="2022-10-22 18:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[18947.509770,18783.619140,18870.109380],
    "high":[18947.50977,18884.25,19004.060547],
    "low":[18679.75,18734.41016,18870.10938],
    "close":[18782.14062,18884.25,18962.26953],
    "abs_body":[165.369150,100.630860,92.16015],
    "body":[-165.369150,100.630860,92.160150]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.769530,46557.179690,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.261720,46075.000000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.871090,45656.550780,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.808590,45655.429690,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.210940,45823.949220,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70013,2022-05-06 00:30:00,39601.148440,39972.550780,39545.058594,39840.53906,NaN,239.390620,239.390620,NaN,NaN
70014,2022-05-06 01:00:00,39844.640620,39903.199220,39671.898440,39816.12109,NaN,28.519530,-28.519530,NaN,NaN
70015,2022-10-22 17:30:00,18947.509770,18947.509770,18679.750000,18782.14062,NaN,165.369150,-165.369150,NaN,NaN
70016,2022-10-22 18:00:00,18783.619140,18884.250000,18734.410160,18884.25000,NaN,100.630860,100.630860,NaN,NaN


In [274]:
missing_times=pd.date_range(start="2022-11-17 04:30",end="2022-11-17 05:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16861.900390,16844.580080,16880.15039],
    "high":[16877.58984,16882.83008,16911.92969],
    "low":[16824.43945,16820.90039,16782.50977],
    "close":[16839.30078,16880.15039,16782.50977],
    "abs_body":[22.599610,35.570310,97.640620],
    "body":[-22.599610,35.570310,-97.640620]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.769530,46557.17969,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.261720,46075.00000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.871090,45656.55078,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.808590,45655.42969,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.210940,45823.94922,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70016,2022-10-22 18:00:00,18783.619140,18884.250000,18734.41016,18884.25000,NaN,100.630860,100.630860,NaN,NaN
70017,2022-10-22 18:30:00,18870.109380,19004.060547,18870.10938,18962.26953,NaN,92.160150,92.160150,NaN,NaN
70018,2022-11-17 04:30:00,16861.900390,16877.589840,16824.43945,16839.30078,NaN,22.599610,-22.599610,NaN,NaN
70019,2022-11-17 05:00:00,16844.580080,16882.830080,16820.90039,16880.15039,NaN,35.570310,35.570310,NaN,NaN


In [275]:
missing_times=pd.date_range(start="2022-11-18 05:00",end="2022-11-18 10:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16844.580080,16880.15039,16681.929690,16688.31055,16697.230470,16684.380860,16653.359380,16584.320310,16580.169920,16508.189450,16470.33008],
    "high":[16882.83008,16911.92969,16711.84961,16727.19922,16710.41992,16691.91016,16672.31055,16593.53906,16582.75977,16509.92969,16519.50977],
    "low":[16820.90039,16782.50977,16660.23047,16681.28906,16671.66992,16642.070312,16582.31055,16568.24023,16498.17969,16403.44922,16467.25],
    "close":[16880.15039,16782.50977,16688.230470,16697.349610,16680.490230,16655.009766,16582.310550,16583.050781,16502.509770,16472.599610,16489.439450],
    "abs_body":[35.570310,97.640620,6.30078,9.03906,16.74024,29.371094,71.04883,1.269529,77.660150,35.589840,19.109370],
    "body":[35.570310,-97.640620,6.300780,9.039060,-16.740240,-29.371094,-71.048830,-1.269529,-77.660150,-35.589840,19.109370]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.531250,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075.00000,46080.871090,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.148440,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.410160,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.539060,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70027,2022-11-18 08:00:00,16653.359380,16672.31055,16582.31055,16582.310550,NaN,71.048830,-71.048830,NaN,NaN
70028,2022-11-18 08:30:00,16584.320310,16593.53906,16568.24023,16583.050781,NaN,1.269529,-1.269529,NaN,NaN
70029,2022-11-18 09:00:00,16580.169920,16582.75977,16498.17969,16502.509770,NaN,77.660150,-77.660150,NaN,NaN
70030,2022-11-18 09:30:00,16508.189450,16509.92969,16403.44922,16472.599610,NaN,35.589840,-35.589840,NaN,NaN


In [276]:
missing_times=pd.date_range(start="2022-11-19 05:00",end="2022-11-19 19:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16844.58008,16880.15039,16681.92969,16688.31055,16697.23047,16684.38086,16653.35938,16584.32031,16580.16992,16508.18945,16470.33008,16825.42969,16810.34961,16791.14062,16744.73047,16778.31055,16767.76953,16727.86914,16711.51953,16715.65039,16727.16992,16730.38086,16781.96094,16750.039062,16736.25977,16737.16016,16754.5293,16759.91992,16777.41992,16742.2793],
    "high":[16882.83008,16911.92969,16711.84961,16727.19922,16710.41992,16691.91016,16672.31055,16593.53906,16582.75977,16509.92969,16519.50977,16825.42969,16815.41016,16806.4707,16788.050781,16785.25977,16770.11914,16731.35938,16721.9707,16740.089844,16745.68945,16789,16833.86914,16769.050781,16746.17969,16764.41992,16771.84961,16798.17969,16796.070312,16765.97070],
    "low":[16820.90039,16782.50977,16660.23047,16681.28906,16671.66992,16642.070312,16582.31055,16568.24023,16498.17969,16403.44922,16467.25,16806.82031,16772.98047,16744.21094,16743.75,16751.63086,16708.80078,16663.88086,16688.91016,16711.58984,16707.10938,16726.96094,16708.39062,16726.63086,16702.25977,16730.90039,16745.53906,16759.91992,16724.74023,16730.83008],
    "close":[16880.15039,16782.50977,16688.23047,16697.34961,16680.49023,16655.009766,16582.31055,16583.050781,16502.50977,16472.59961,16489.43945,16806.82031,16789.84961,16749.4707,16778.58984,16770.11914,16731.13086,16712.51953,16714.67969,16728.48047,16729.35938,16775.28906,16756.53906,16737.25,16737.83984,16753.91992,16764.2207,16777.070312,16744.66016,16759.94922],
    "abs_body":[35.57031,97.64062,6.30078,9.03906,16.74024,29.371094,71.04883,1.269529,77.66015,35.58984,19.10937,18.609380,20.5,41.66992,33.85937,8.19141,36.63867,15.34961,3.16016,12.83008,2.18946,44.9082,25.42188,12.789062,1.58007,16.75976,9.6914,17.150392,32.75976,17.66992],
    "body":[35.57031,-97.64062,6.30078,9.03906,-16.74024,-29.371094,-71.04883,-1.269529,-77.66015,-35.58984,19.10937,-18.609380,-20.5,-41.66992,33.85937,-8.19141,-36.63867,-15.34961,3.16016,12.83008,2.18946,44.9082,-25.42188,-12.789062,1.58007,16.75976,9.6914,17.150392,-32.75976,17.66992]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.769530,46557.17969,46640.531250,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.261720,46075.00000,46080.871090,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.871090,45656.55078,45693.148440,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.808590,45655.42969,46037.410160,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.210940,45823.94922,45881.539060,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70057,2022-11-19 17:30:00,16737.160160,16764.419920,16730.90039,16753.919920,NaN,16.759760,16.759760,NaN,NaN
70058,2022-11-19 18:00:00,16754.529300,16771.849610,16745.53906,16764.220700,NaN,9.691400,9.691400,NaN,NaN
70059,2022-11-19 18:30:00,16759.919920,16798.179690,16759.91992,16777.070312,NaN,17.150392,17.150392,NaN,NaN
70060,2022-11-19 19:00:00,16777.419920,16796.070312,16724.74023,16744.660160,NaN,32.759760,-32.759760,NaN,NaN


In [277]:
missing_times=pd.date_range(start="2022-11-20 14:30",end="2022-11-20 15:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16715.65039,16727.16992],
    "high":[16740.089844,16745.68945],
    "low":[16711.58984,16707.10938],
    "close":[16728.48047,16729.35938],
    "abs_body":[12.83008,2.18946],
    "body":[12.83008,2.18946]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.769530,46557.17969,46640.531250,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.261720,46075.00000,46080.871090,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.871090,45656.55078,45693.148440,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.808590,45655.42969,46037.410160,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.210940,45823.94922,45881.539060,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70059,2022-11-19 18:30:00,16759.919920,16798.179690,16759.91992,16777.070312,NaN,17.150392,17.150392,NaN,NaN
70060,2022-11-19 19:00:00,16777.419920,16796.070312,16724.74023,16744.660160,NaN,32.759760,-32.759760,NaN,NaN
70061,2022-11-19 19:30:00,16742.279300,16765.970700,16730.83008,16759.949220,NaN,17.669920,17.669920,NaN,NaN
70062,2022-11-20 14:30:00,16715.650390,16740.089844,16711.58984,16728.480470,NaN,12.830080,12.830080,NaN,NaN


In [278]:
missing_times=pd.date_range(start="2022-11-20 16:00",end="2022-11-20 16:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16730.38086],
    "high":[16789],
    "low":[16726.96094],
    "close":[16775.28906],
    "abs_body":[44.9082],
    "body":[44.9082]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.769530,46557.17969,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.261720,46075.00000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.871090,45656.55078,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.808590,45655.42969,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.210940,45823.94922,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70060,2022-11-19 19:00:00,16777.419920,16796.070312,16724.74023,16744.66016,NaN,32.759760,-32.759760,NaN,NaN
70061,2022-11-19 19:30:00,16742.279300,16765.970700,16730.83008,16759.94922,NaN,17.669920,17.669920,NaN,NaN
70062,2022-11-20 14:30:00,16715.650390,16740.089844,16711.58984,16728.48047,NaN,12.830080,12.830080,NaN,NaN
70063,2022-11-20 15:00:00,16727.169920,16745.689450,16707.10938,16729.35938,NaN,2.189460,2.189460,NaN,NaN


In [279]:
missing_times=pd.date_range(start="2022-12-03 13:00",end="2022-12-03 16:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[16932.40039,16958.76953,16948.029297,16967.91016,16965.92969,16967.32031,16954.35938,16967.060547],
    "high":[16970.65039,16965.75977,16972.85938,16978.36914,16988.4707,16977.070312,16968.7207,16995.019531],
    "low":[16932.40039,16941.59961,16948.029297,16960.82031,16964.58984,16952.78906,16951.17969,16959.36914],
    "close":[16965.30078,16947.4707,16966.25977,16966.46094,16966.029297,16954.82031,16966.88086,16974.080078],
    "abs_body":[32.90039,11.29883,18.230473,1.44922,0.099607,12.5,12.52148,7.019531],
    "body":[32.90039,-11.29883,-18.230473,-1.44922,0.099607,-12.5,12.52148,7.019531]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.769530,46557.17969,46640.531250,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.261720,46075.00000,46080.871090,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.871090,45656.55078,45693.148440,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.808590,45655.42969,46037.410160,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.210940,45823.94922,45881.539060,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70068,2022-12-03 14:30:00,16967.910160,16978.369140,16960.82031,16966.460940,NaN,1.449220,-1.449220,NaN,NaN
70069,2022-12-03 15:00:00,16965.929690,16988.470700,16964.58984,16966.029297,NaN,0.099607,0.099607,NaN,NaN
70070,2022-12-03 15:30:00,16967.320310,16977.070312,16952.78906,16954.820310,NaN,12.500000,-12.500000,NaN,NaN
70071,2022-12-03 16:00:00,16954.359380,16968.720700,16951.17969,16966.880860,NaN,12.521480,12.521480,NaN,NaN


In [ ]:
missing_times=pd.date_range(start="2023-02-22 21:30",end="2023-02-23 11:30",freq="30min")
missing_df=pd.DataFrame({ 
    "datetime":missing_times,
    "open":[24563.85938,24381.33984,24409.80078,24401.59961,24641.91016,24629.23047,24689.48047,24640.019531,24607.7207,24449.66992,24459.94922,24241.070312,24190.16992,24295.82031,24389,24361.81055,24452.98047,24412.73047,24403.23047,24360.14062,24203.83984,24036.24023,24172.31055,24229.11914,24201.96094,24163.82031,24158.83984,24026.33008,24102.78906],
    "high":[24587.75977,24444.42969,24444.65039,24652.85938,24746.90039,24732.019531,24693.90039,24686.53906,24642.88086,24525.67969,24536.65039,24332.26953,24316.30078,24428.84961,24404.33984,24451.90039,24474.51953,24425.5293,24432.75,24392.070312,24203.83984,24195.30078,24270.40039,24247.11914,24222.73047,24170.32031,24168.15039,24113.029297,24128.25977],
    "low":[24321.32031,24301.5293,24380.76953,24401.59961,24589.96094,24569.58008,24564.4707,24558.17969,24450.56055,24434.57031,24159.85938,24173.41016,24174.28906,24280.82031,24341.019531,24351.32031,24411.49023,24276.41016,24356.019531,24156.5,23969.66016,23873.0097656,24133.11914,24155.86914,24160.029297,24031.099609,24007.91992,24013.31055,24028.17969],
    "close":[24377.19922,24403.060547,24412.69922,24652.85938,24631.66016,24693.90039,24636.63086,24595.96094,24450.88086,24464.41016,24257.78906,24175.57031,24295.33984,24384.94922,24356.9707,24451.90039,24411.83984,24402.7793,24365.51953,24195.76953,24036.60938,24169.50977,24238.68945,24200.099609,24162.17969,24161.21094,24032.81055,24107.34961,24044.48047],
    "abs_body":[186.66016,21.720707,2.89844,251.25977,10.25,64.66992,52.84961,44.058591,156.83984,14.74024,202.16016,65.500002,105.16992,89.12891,32.0293,90.08984,41.14063,9.95117,37.71094,164.37109,167.23046,133.26954,66.3789,29.019531,39.78125,2.60937,126.02929,81.01953,58.30859],
    "body":[-186.66016,21.720707,2.89844,251.25977,-10.25,64.66992,-52.84961,-44.058591,-156.83984,14.74024,-202.16016,-65.500002,105.16992,89.12891,-32.0293,90.08984,-41.14063,-9.95117,-37.71094,-164.37109,-167.23046,133.26954,66.3789,-29.019531,-39.78125,-2.60937,-126.02929,81.01953,-58.30859]   
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.769530,46557.179690,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.261720,46075.000000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.871090,45656.550780,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.808590,45655.429690,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.210940,45823.949220,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70097,2023-02-23 09:30:00,24201.960940,24222.730470,24160.029297,24162.17969,NaN,39.781250,-39.781250,NaN,NaN
70098,2023-02-23 10:00:00,24163.820310,24170.320310,24031.099609,24161.21094,NaN,2.609370,-2.609370,NaN,NaN
70099,2023-02-23 10:30:00,24158.839840,24168.150390,24007.919920,24032.81055,NaN,126.029290,-126.029290,NaN,NaN
70100,2023-02-23 11:00:00,24026.330080,24113.029297,24013.310550,24107.34961,NaN,81.019530,81.019530,NaN,NaN


In [281]:
missing_times=pd.date_range(start="2023-03-04 23:00",end="2023-03-05 01:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[22422.019531,22420.60938,22386.86914,22314.14062,22306.85938,22354.26953],
    "high":[22448.36914,22422.050781,22394.24023,22355.99023,22389.84961,22377.71094],
    "low":[22395.92969,22375.25,22263.57031,22294.13086,22301.080078,22333.2207],
    "close":[22417.42969,22394.24023,22316.21094,22311.89062,22356.48047,22355.060547],
    "abs_body":[4.589841,26.36915,70.6582,2.25,49.62109,0.791017],
    "body":[-4.589841,-26.36915,-70.6582,-2.25,49.62109,0.791017]
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.769530,46557.179690,46640.531250,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.261720,46075.000000,46080.871090,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.871090,45656.550780,45693.148440,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.808590,45655.429690,46037.410160,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.210940,45823.949220,45881.539060,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70103,2023-03-04 23:30:00,22420.609380,22422.050781,22375.250000,22394.240230,NaN,26.369150,-26.369150,NaN,NaN
70104,2023-03-05 00:00:00,22386.869140,22394.240230,22263.570310,22316.210940,NaN,70.658200,-70.658200,NaN,NaN
70105,2023-03-05 00:30:00,22314.140620,22355.990230,22294.130860,22311.890620,NaN,2.250000,-2.250000,NaN,NaN
70106,2023-03-05 01:00:00,22306.859380,22389.849610,22301.080078,22356.480470,NaN,49.621090,49.621090,NaN,NaN


In [282]:
missing_times=pd.date_range(start="2023-07-20 00:00",end="2023-07-20 03:30",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[29929.14,29909.63,29775.01,29720.64,29728.11,29783.77,29830.69,29807.85],
    "high":[29932.4,29946.23,29817.33,29794.65,29818.44,29841.05,29831.53,29823.14],
    "low":[29859.69,29771.55,29685.27,29704.95,29690.39,29756.01,29778.89,29768.22],
    "close":[29910.96,29775,29722.14,29728.11,29783.61,29830.7,29807.88,29790.82],
    "abs_body":[18.18,134.63,52.87,7.47,55.5,46.93,22.81,17.03],
    "body":[-18.18,-134.63,-52.87,7.47,55.5,46.93,-22.81,-17.03]
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075.00000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70111,2023-07-20 01:30:00,29720.640000,29794.65000,29704.95000,29728.11000,NaN,7.470000,7.470000,NaN,NaN
70112,2023-07-20 02:00:00,29728.110000,29818.44000,29690.39000,29783.61000,NaN,55.500000,55.500000,NaN,NaN
70113,2023-07-20 02:30:00,29783.770000,29841.05000,29756.01000,29830.70000,NaN,46.930000,46.930000,NaN,NaN
70114,2023-07-20 03:00:00,29830.690000,29831.53000,29778.89000,29807.88000,NaN,22.810000,-22.810000,NaN,NaN


In [283]:
missing_times=pd.date_range(start="2024-10-26 22:00",end="2024-10-26 22:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[67579.54],
    "high":[67864.16],
    "low":[67512.9],
    "close":[67633.74],
    "abs_body":[54.2],
    "body":[54.2]
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075.00000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70112,2023-07-20 02:00:00,29728.110000,29818.44000,29690.39000,29783.61000,NaN,55.500000,55.500000,NaN,NaN
70113,2023-07-20 02:30:00,29783.770000,29841.05000,29756.01000,29830.70000,NaN,46.930000,46.930000,NaN,NaN
70114,2023-07-20 03:00:00,29830.690000,29831.53000,29778.89000,29807.88000,NaN,22.810000,-22.810000,NaN,NaN
70115,2023-07-20 03:30:00,29807.850000,29823.14000,29768.22000,29790.82000,NaN,17.030000,-17.030000,NaN,NaN


In [284]:
missing_times=pd.date_range(start="2025-10-25 21:00",end="2025-10-26 02:00",freq="30min")
missing_df=pd.DataFrame({
    "datetime":missing_times,
    "open":[110608.48,110070.08,110370.87,110282.04,110516.01,110231.46,110528.01,110621.56,110948.65,110624.01,110754],
    "high":[110636.73,110411.24,110378.96,110600,110519.16,110658.51,110654.51,110980.7,110955.64,110817.37,111040.55],
    "low":[109910.01,109804.93,109930,110195.8,110231.47,110165.84,110422,110617.05,110506.46,110485.4,110676.22],
    "close":[110070.09,110374.01,110286.93,110516.01,110233.73,110524.36,110621.56,110948.65,110624.14,110753.99,110920.22],
    "abs_body":[538.39,303.93,83.94,233.97,282.28,292.9,93.55,327.09,324.51,129.98,166.22],
    "body":[-538.39,303.93,-83.94,233.97,-282.28,292.9,93.55,327.09,-324.51,129.98,166.22]
})
data=pd.concat([data,missing_df],ignore_index=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075.00000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70123,2025-10-26 00:00:00,110528.010000,110654.51000,110422.00000,110621.56000,NaN,93.550000,93.550000,NaN,NaN
70124,2025-10-26 00:30:00,110621.560000,110980.70000,110617.05000,110948.65000,NaN,327.090000,327.090000,NaN,NaN
70125,2025-10-26 01:00:00,110948.650000,110955.64000,110506.46000,110624.14000,NaN,324.510000,-324.510000,NaN,NaN
70126,2025-10-26 01:30:00,110624.010000,110817.37000,110485.40000,110753.99000,NaN,129.980000,129.980000,NaN,NaN


In [285]:
data['datetime']=pd.to_datetime(data['datetime'])
data=data.sort_values("datetime").reset_index(drop=True)
data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,0.0,253.769530,-253.769530,57.468750,83.35156
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075.00000,46080.87109,0.0,575.390630,-575.390630,0.000000,5.87109
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,0.0,370.921872,-370.921872,254.800778,36.59766
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,1.0,365.679690,365.679690,158.398430,16.30078
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,0.0,160.480471,-160.480471,73.191409,57.58984
...,...,...,...,...,...,...,...,...,...,...
70123,2025-12-31 21:30:00,87830.030000,87932.17000,87558.00000,87792.55000,0.0,37.480000,-37.480000,102.140000,234.55000
70124,2025-12-31 22:00:00,87797.910000,87813.02000,87259.01000,87531.06000,0.0,266.850000,-266.850000,15.110000,272.05000
70125,2025-12-31 22:30:00,87531.060000,87637.99000,87356.29000,87496.24000,0.0,34.820000,-34.820000,106.930000,139.95000
70126,2025-12-31 23:00:00,87496.250000,87550.34000,87400.76000,87487.96000,0.0,8.290000,-8.290000,54.090000,87.20000


In [8]:
df_volume=[]
df_takervolume=[]
df_opentime=[]
df_trades=[]
interval='30m'
for yr in range(2022,2026):
    for m in range(1,13): 
        start_date=f'{yr}-{(m):02d}-01'
        end_date=f'{yr}-{(m):02d}-16'
        start_ms=int(time.mktime(time.strptime(start_date,"%Y-%m-%d")))*1000
        end_ms=int(time.mktime(time.strptime(end_date,"%Y-%m-%d")))*1000
        api_url=f"https://api1.binance.com/api/v3/klines?symbol=BTCUSDT&interval={interval}&startTime={start_ms}&endTime={end_ms}&limit=1000"
        while True:
            session=requests.Session()
            retries=Retry(total=5,backoff_factor=1,status_forcelist=[429,500,502,503,504])
            session.mount("https://",HTTPAdapter(max_retries=retries))
            fetch=session.get(api_url).json()
            if fetch:
                open_time=[value[0] for value in fetch]
                volume=[float(value[7]) for value in fetch]
                taker_volume=[float(value[10]) for value in fetch]
                trades=[float(value[8]) for value in fetch]
                df_volume.extend(volume)
                df_opentime.extend(open_time) 
                df_takervolume.extend(taker_volume)
                df_trades.extend(trades)
                time.sleep(4)
                break
            else: 
                print("API Error:",fetch)
                fetch=None
                break
        last_day=calendar.monthrange(yr,m)[1]
        start_date=f'{yr}-{(m):02d}-16'
        end_date=f'{yr}-{(m):02d}-{(last_day)}'
        start_ms=int(time.mktime(time.strptime(start_date,"%Y-%m-%d")))*1000
        end_ms=int(time.mktime(time.strptime(end_date,"%Y-%m-%d")))*1000
        api_url=f"https://api1.binance.com/api/v3/klines?symbol=BTCUSDT&interval={interval}&startTime={start_ms}&endTime={end_ms}&limit=1000"
        while True:
            session=requests.Session()
            retries=Retry(total=5,backoff_factor=1,status_forcelist=[429,500,502,503,504])
            session.mount("https://",HTTPAdapter(max_retries=retries))
            fetch=session.get(api_url).json()
            if fetch:
                open_time=[value[0] for value in fetch]
                volume=[float(value[7]) for value in fetch]
                taker_volume=[float(value[10]) for value in fetch]
                trades=[float(value[8]) for value in fetch]
                df_volume.extend(volume)
                df_opentime.extend(open_time) 
                df_takervolume.extend(taker_volume)
                df_trades.extend(trades)
                time.sleep(4)
                break
            else: 
                print("API Error:",fetch)
                fetch=None
                break
df=pd.DataFrame({
    'open_time':pd.to_datetime(df_opentime,unit='ms'),
    'volume':df_volume,
    'taker_volume':df_takervolume,
    'no_of_trades':df_trades
}).dropna()
df['open_time']=df['open_time'].dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata')
df

,open_time,volume,taker_volume,no_of_trades
0,2022-01-01 01:00:00+05:30,1.146748e+08,4.930189e+07,49761.0
1,2022-01-01 01:30:00+05:30,6.113717e+07,3.375046e+07,32543.0
2,2022-01-01 02:00:00+05:30,2.308425e+07,1.213224e+07,18757.0
3,2022-01-01 02:30:00+05:30,4.087805e+07,2.338065e+07,27016.0
4,2022-01-01 03:00:00+05:30,3.909260e+07,2.144366e+07,27563.0
...,...,...,...,...
67913,2025-12-30 23:00:00+05:30,4.250846e+07,2.167012e+07,105824.0
67914,2025-12-30 23:30:00+05:30,1.873078e+07,1.009447e+07,72103.0
67915,2025-12-31 00:00:00+05:30,1.963927e+07,9.808946e+06,63776.0
67916,2025-12-31 00:30:00+05:30,2.163632e+07,1.154721e+07,93779.0


In [9]:
df.to_csv('binance_22-25.csv',index=False)

In [3]:
import time
time.time()

1774947119.4192133

In [5]:
import requests
def get_binance_server_time():  
    response=requests.get("https://api1.binance.com/api/v3/time")
    return response.json()['serverTime']
get_binance_server_time()

1774943942379

In [ ]:
merged_data=pd.merge_asof(data,df,left_index=True,right_index=True,direction='nearest',tolerance=60)
merged_data

,datetime,open,high,low,close,color,abs_body,body,upper_wick,lower_wick,open_time,volume,taker_volume,no_of_trades
0,2022-01-01 00:00:00,46894.300780,46951.76953,46557.17969,46640.53125,0,253.769530,-253.769530,57.468750,83.35156,2021-12-31 18:30:00,5.741317e+07,2.218630e+07,33022.0
1,2022-01-01 00:30:00,46656.261720,46656.26172,46075.00000,46080.87109,0,575.390630,-575.390630,0.000000,5.87109,2021-12-31 19:00:00,6.294230e+07,2.873560e+07,37711.0
2,2022-01-01 01:00:00,46064.070312,46318.87109,45656.55078,45693.14844,0,370.921872,-370.921872,254.800778,36.59766,2021-12-31 19:30:00,1.146748e+08,4.930189e+07,49761.0
3,2022-01-01 01:30:00,45671.730470,46195.80859,45655.42969,46037.41016,1,365.679690,365.679690,158.398430,16.30078,2021-12-31 20:00:00,6.113717e+07,3.375046e+07,32543.0
4,2022-01-01 02:00:00,46042.019531,46115.21094,45823.94922,45881.53906,0,160.480471,-160.480471,73.191409,57.58984,2021-12-31 20:30:00,2.308425e+07,1.213224e+07,18757.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69240,2025-12-30 22:00:00,89136.030000,89190.74000,88484.70000,88659.99000,0,476.040000,-476.040000,54.710000,175.29000,NaT,NaN,NaN,NaN
69241,2025-12-30 22:30:00,88660.000000,88839.49000,88414.06000,88549.49000,0,110.510000,-110.510000,179.490000,135.43000,NaT,NaN,NaN,NaN
69242,2025-12-30 23:00:00,88546.190000,88613.98000,88050.17000,88297.59000,0,248.600000,-248.600000,67.790000,247.42000,NaT,NaN,NaN,NaN
69243,2025-12-30 23:30:00,88297.580000,88386.68000,88203.77000,88304.86000,1,7.280000,7.280000,81.820000,93.81000,NaT,NaN,NaN,NaN


In [ ]:
merged_data['volume']=pd.to_numeric(merged_data['volume'])
merged_data['taker_volume']=pd.to_numeric(merged_data['taker_volume'])
merged_data['no_of_trades']=pd.to_numeric(merged_data['no_of_trades'])
merged_data['taker_buy_ratio']=merged_data['taker_volume']/merged_data['volume']
merged_data['avg_trade_size']=merged_data['volume']/merged_data['no_of_trades']
merged_data['vol_z']=(merged_data['volume']-merged_data['volume'].rolling(48).mean().shift(1))/merged_data['volume'].rolling(48).std().shift(1)

In [ ]:
data=merged_data
data

NameError: name 'merged_data' is not defined

In [251]:
pfvg=0
nfvg=0
for i in range(1,len(data)-1):
    if data['low'][i+1]-data['high'][i-1]>25:
        pfvg+=1
    elif data['low'][i-1]-data['high'][i+1]>25:
        nfvg+=1
pfvg,nfvg

(4776, 4390)

In [452]:
fvgs=[25,50,75,100,150,200,250,300,350]
fvgs=[0.001,0.002,0.003,0.004,0.006,0.008,0.01,0.012,0.014]
horizons=[4,6,8,10,12,14,16,18,20,22,24,26,28]
for f in fvgs:
    for h in horizons:
        pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
        for i in range(1,len(data)-h):
            if (data['low'][i+1]-data['high'][i-1])/data['close'][i]>f:
                pfvg_ret+=data['close'][i+h]-data['close'][i]
                pfvg+=1
                pret_per+=(data['close'][i+h]-data['close'][i])/data['close'][i]
            elif (data['low'][i-1]-data['high'][i+1])/data['close'][i]>f:
                nfvg_ret+=data['close'][i]-data['close'][i+h]
                nfvg+=1
                nret_per+=(data['close'][i]-data['close'][i+h])/data['close'][i]
        print(f'fair value gap {f}; horizon {h}: ',pfvg_ret/pfvg,nfvg_ret/nfvg,pret_per*100/pfvg,nret_per*100/nfvg,pfvg,nfvg)

fair value gap 0.001; horizon 4:  116.93878762432 113.85695868337164 0.2228462146341747 0.1981778077870691 3676 3464
fair value gap 0.001; horizon 6:  120.71049589162101 114.42150379601591 0.23678358934191276 0.19380607073933848 3676 3464
fair value gap 0.001; horizon 8:  123.46664056158893 112.12187475644123 0.24166608773334214 0.1903300617194902 3676 3462
fair value gap 0.001; horizon 10:  119.6066200321548 114.38781360346609 0.2299538530186649 0.1884196523005446 3676 3462
fair value gap 0.001; horizon 12:  116.70699467328598 115.095160696476 0.2352324274946461 0.1929776137118516 3676 3462
fair value gap 0.001; horizon 14:  115.33141911828059 113.5795224079142 0.23654063640412246 0.1853639700315795 3676 3462
fair value gap 0.001; horizon 16:  124.3375604186344 113.68288857654535 0.25495720593780863 0.1863715719259863 3676 3462
fair value gap 0.001; horizon 18:  135.94189025952122 110.07587507365696 0.27787808686420395 0.1805160130210455 3676 3462
fair value gap 0.001; horizon 20:  12

In [253]:
pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
pret=[]
nret=[]
for i in range(1,len(data)-16):
    if data['low'][i+1]-data['high'][i-1]>25:
        pfvg_ret+=data['close'][i+16]-data['close'][i]
        pfvg+=1
        pret_per+=(data['close'][i+16]-data['close'][i])/data['close'][i]
        pret.append(pfvg_ret)
    elif data['low'][i-1]-data['high'][i+1]>25:
        nfvg_ret+=data['close'][i]-data['close'][i+16]
        nfvg+=1
        nret_per+=(data['close'][i]-data['close'][i+16])/data['close'][i]
        nret.append(nfvg_ret)

In [254]:
nret

[-725.3007900000011,
 -669.1015700000062,
 -1061.9531300000017,
 -772.53125,
 -825.5390599999955,
 -643.8124949999983,
 -559.1601559999981,
 -665.9296859999959,
 -1222.160155999998,
 -143.7070259999964,
 52.05469400000584,
 -519.8945259999964,
 130.42187399999966,
 1730.5820329999988,
 4164.812492999998,
 6014.6328039999935,
 7343.453114999989,
 7893.816394999994,
 8514.917964999993,
 9129.335934999996,
 9157.164064999997,
 8702.886724999997,
 8298.664074999993,
 8187.683604999991,
 7799.835954999995,
 9048.316424999997,
 8858.867204999995,
 8432.328145,
 8319.839865000002,
 8605.128925000005,
 8409.976575,
 9675.988295000003,
 9766.207045000003,
 9025.609385000003,
 8315.492194000006,
 7594.472664000008,
 6702.98438400001,
 6349.816414000008,
 6164.804694000006,
 7263.1953240000075,
 8194.175794000002,
 8871.734394,
 9067.023454000002,
 8601.605484000007,
 8102.957054000006,
 7130.929714000005,
 7051.601584000004,
 7153.820344000007,
 7062.632844000007,
 6258.703164000006,
 6523.00003

In [460]:
# Unmitigated
horizons=[4,6,8,10,12,14,16,20,24,28,32]
for h in horizons:
    pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
    pret=[]
    nret=[]
    for i in range(1,len(data)-8):
        if (data['low'][i+1]-data['high'][i-1])/data['close'][i]>0.001:
            mitigated=False 
            for j in range(i+2,i+h):
                if data['close'][j]<data['low'][i+1]:
                    mitigated=True
                    break
            if not mitigated:
                pfvg_ret+=data['close'][i+16]-data['close'][i]
                pfvg+=1
                pret_per+=(data['close'][i+16]-data['close'][i])/data['close'][i]
                pret.append(data['close'][i+16]-data['close'][i])
        elif (data['low'][i-1]-data['high'][i+1])/data['close'][i]>0.001:
            mitigated=False
            for j in range(i+2,i+h):
                if data['close'][j]>data['high'][i+1]:
                    mitigated=True
                    break
            if not mitigated:
                nfvg_ret+=data['close'][i]-data['close'][i+16]
                nfvg+=1
                nret_per+=(data['close'][i]-data['close'][i+16])/data['close'][i]
                nret.append(data['close'][i]-data['close'][i+16])
    print(f'horizon {h}: ',pfvg_ret/pfvg,nfvg_ret/nfvg,pret_per*100/pfvg,nret_per*100/nfvg,pfvg,nfvg,sum(1 for x in pret if x<0)/pfvg,sum(1 for x in nret if x<0)/nfvg)

horizon 4:  299.36368867519934 314.15628748649124 0.5953654227324817 0.5581314249006045 2504 2243 0.34584664536741216 0.3727151136870263
horizon 6:  425.0037559535995 446.5227701803921 0.8401778238889883 0.8154498892010614 2028 1785 0.28155818540433925 0.29523809523809524
horizon 8:  513.0338863164101 549.5959360562058 1.0102597569456968 1.004187902365958 1755 1555 0.23304843304843303 0.24115755627009647
horizon 10:  603.0455903199223 634.7759855895266 1.1767549017679 1.1576216865337747 1556 1394 0.18766066838046272 0.18794835007173602
horizon 12:  682.0237456088912 695.0713078319001 1.3201951230405784 1.2753637905499808 1417 1279 0.14114326040931546 0.1485535574667709
horizon 14:  747.3116527907569 748.2794617127487 1.4435796852462617 1.3700880369027182 1320 1208 0.10681818181818181 0.11423841059602649
horizon 16:  828.7635069452317 821.406885659768 1.6027616520361954 1.5101843599666236 1227 1121 0.05378973105134474 0.06333630686886708
horizon 20:  900.1598530391766 887.6185854913174 

In [441]:
pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
pret=[]
nret=[]
for i in range(1,len(data)-16):
    if data['low'][i+1]-data['high'][i-1]>25:
        mitigated=False 
        for j in range(i+2,i+8):
            if data['close'][j]<data['low'][i+1]:
                mitigated=True
                break
        if not mitigated:
            pfvg_ret+=data['close'][i+16]-data['close'][i]
            pfvg+=1
            pret_per+=(data['close'][i+16]-data['close'][i])/data['close'][i]
            pret.append(pfvg_ret)
    elif data['low'][i-1]-data['high'][i+1]>25:
        mitigated=False
        for j in range(i+2,i+8):
            if data['close'][j]>data['high'][i+1]:
                mitigated=True
                break
        if not mitigated:
            nfvg_ret+=data['close'][i]-data['close'][i+16]
            nfvg+=1
            nret_per+=(data['close'][i]-data['close'][i+16])/data['close'][i]
            nret.append(nfvg_ret)

In [442]:
sum(1 for x in pret if x<0)

0

In [ ]:
# Consolidating 
pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
pret=[]
nret=[]
for i in range(1,len(data)-24):
    if data['low'][i+1]-data['high'][i-1]>300:
        consolidating=False
        if data['abs_body'][i+1]<50:
            consolidating=True 
        if consolidating:
            pfvg_ret+=data['close'][i+24]-data['close'][i]
            pfvg+=1
            pret_per+=(data['close'][i+24]-data['close'][i])/data['close'][i]
            pret.append(pfvg_ret)
    elif data['low'][i-1]-data['high'][i+1]>200:
        consolidating=False
        if data['abs_body'][i+1]<25:
            consolidating=True
        if consolidating:
            nfvg_ret+=data['close'][i]-data['close'][i+12]
            nfvg+=1
            nret_per+=(data['close'][i]-data['close'][i+12])/data['close'][i]
            nret.append(nfvg_ret)

In [377]:
pfvg_ret/pfvg,nfvg_ret/nfvg,pfvg,nfvg

(45.25647218185087, 195.13343547435966, 1113, 78)

In [286]:
sum(1 for x in nret if x<0)

0

In [487]:
# Inversion FVG
pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
pret=[]
nret=[]
for i in range(1,len(data)-12):
    if data['low'][i+1]-data['high'][i-1]>100:
        ifvg=False
        for j in range(i+4,i+12):
            if data['low'][j-1]-data['high'][j+1]>200:
                ifvg=True
                break
        if ifvg:
            pfvg_ret+=data['close'][i+24]-data['close'][j+1]
            pfvg+=1
            pret_per+=(data['close'][i+24]-data['close'][j+1])/data['close'][j+1]
            pret.append(pfvg_ret)
    elif data['low'][i-1]-data['high'][i+1]>100:
        ifvg=False
        for j in range(i+4,i+12):
            if data['low'][j+1]-data['high'][j-1]>200:
                ifvg=True 
                break
        if ifvg:
            nfvg_ret+=data['close'][j+1]-data['close'][i+24]
            nfvg+=1
            nret_per+=(data['close'][j+1]-data['close'][i+24])/data['close'][j+1]
            nret.append(nfvg_ret)

In [488]:
pfvg_ret/pfvg,nfvg_ret/nfvg,pfvg,nfvg

(118.81598058263343, 90.28549785770473, 357, 305)

In [ ]:
sum(1 for x in pret if x<0)

21

In [491]:
# Reaction
horizons=[2,4,8,12,16,20,24,28,32]
for h in horizons:
    pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
    for i in range(1,len(data)-h):
        if data['low'][i+1]-data['high'][i-1]>25:
            reaction=True  
            for j in range(i+2,i+h):
                if data['close'][j]<data['high'][i-1]:
                    reaction=False
            if reaction:
                pfvg_ret+=data['close'][i+h]-data['close'][i]
                pfvg+=1
                pret_per+=(data['close'][i+h]-data['close'][i])/data['close'][i]
        elif data['low'][i-1]-data['high'][i+1]>25:
            reaction=True 
            for j in range(i+2,i+h):
                if data['close'][j]>data['low'][i-1]:
                    reaction=False
            if reaction:
                nfvg_ret+=data['close'][i]-data['close'][i+h]
                nfvg+=1
                nret_per+=(data['close'][i]-data['close'][i+h])/data['close'][i]
    print(f'horizon {h}: ',pfvg_ret/pfvg,nfvg_ret/nfvg,pret_per*100/pfvg,nret_per*100/nfvg,pfvg,nfvg)

horizon 2:  105.73175283322213 110.15522198866215 0.18737913964590383 0.18996211693836104 4795 4410
horizon 4:  197.56004936684138 210.94723785323865 0.3443719565457301 0.3581326490137927 4005 3659
horizon 8:  375.1844627429502 389.78554742069196 0.6695932629358876 0.6633831923072663 3064 2774
horizon 12:  526.9493965451913 537.8739066951125 0.935385767295921 0.9111296512850424 2558 2332
horizon 16:  670.1179091802916 645.5924304545277 1.1905380583174903 1.0900603735873093 2263 2054
horizon 20:  771.7594435463272 755.5414067481751 1.3783117487568513 1.2821614012210065 2055 1864
horizon 24:  880.9672990447233 873.3168351237914 1.567135162616811 1.4974518229622327 1914 1719
horizon 28:  970.8598184281263 942.9802083860244 1.7496747402046802 1.645521692466411 1799 1603
horizon 32:  1063.4874505244763 1003.1338143819273 1.9112239996718667 1.7685447470829259 1716 1494


In [492]:
pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
pret=[]
nret=[]
for i in range(1,len(data)-4):
    if data['low'][i+1]-data['high'][i-1]>25:
        reaction=True  
        for j in range(i+2,i+4):
            if data['close'][j]<data['high'][i-1]:
                reaction=False
        if reaction:
            pfvg_ret+=data['close'][i+4]-data['close'][i]
            pfvg+=1
            pret_per+=(data['close'][i+4]-data['close'][i])/data['close'][i]
            pret.append(pfvg_ret)
    elif data['low'][i-1]-data['high'][i+1]>25:
        reaction=True 
        for j in range(i+2,i+4):
            if data['close'][j]>data['low'][i-1]:
                reaction=False
        if reaction:
            nfvg_ret+=data['close'][i]-data['close'][i+4]
            nfvg+=1
            nret_per+=(data['close'][i]-data['close'][i+4])/data['close'][i]
            nret.append(nfvg_ret)

In [495]:
len(pret)

4005

In [496]:
sum(1 for x in pret if x<0)

7

In [1]:
# Support/Resistance
horizons=[4,5,6,7,8]
for h in horizons:
    pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
    for i in range(h,len(data)-h):
        if data['low'][i+1]-data['high'][i-1]>25:
            support=False
            for j in range(2,h):
                if data['high'][i-1]<data['high'][i-j]<data['low'][i+1]:
                    support=True
                if data['high'][i-j]>data['low'][i+1]:
                    support=False
                    break
            if support:
                pfvg_ret+=data['close'][i+h]-data['close'][i]
                pfvg+=1
                pret_per+=(data['close'][i+h]-data['close'][i])/data['close'][i]
        elif data['low'][i-1]-data['high'][i+1]>25:
            resistance=False
            for j in range(2,h):
                if data['high'][i+1]<data['low'][i-j]<data['low'][i-1]:
                    resistance=True 
                if data['low'][i-j]<data['high'][i+1]:
                    resistance=False
                    break
            if resistance:
                nfvg_ret+=data['close'][i]-data['close'][i+h]
                nfvg+=1
                nret_per+=(data['close'][i]-data['close'][i+h])/data['close'][i]
    print(f'horizon {h}: ',pfvg_ret/pfvg,nfvg_ret/nfvg,pret_per*100/pfvg,nret_per*100/nfvg,pfvg,nfvg)

NameError: name 'data' is not defined

In [499]:
pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
pret=[]
nret=[]
for i in range(7,len(data)-7):
    if data['low'][i+1]-data['high'][i-1]>25:
        support=False
        for j in range(2,5):
            if data['high'][i-1]<data['high'][i-j]<data['low'][i+1]:
                support=True
            if data['high'][i-j]>data['low'][i+1]:
                support=False
                break
        if support:
            pfvg_ret+=data['close'][i+5]-data['close'][i]
            pfvg+=1
            pret_per+=(data['close'][i+5]-data['close'][i])/data['close'][i]
            pret.append(pfvg)
    elif data['low'][i-1]-data['high'][i+1]>25:
        resistance=False
        for j in range(2,7):
            if data['high'][i+1]<data['low'][i-j]<data['low'][i-1]:
                resistance=True 
            if data['low'][i-j]<data['high'][i+1]:
                resistance=False
                break
        if resistance:
            nfvg_ret+=data['close'][i]-data['close'][i+7]
            nfvg+=1
            nret_per+=(data['close'][i]-data['close'][i+7])/data['close'][i]
            nret.append(nfvg)

In [501]:
sum(1 for x in nret if x<0)

0

In [267]:
# Priority
for h in range(1,21):
    pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
    i=1
    while i<len(data)-20:
        if data['low'][i+1]-data['high'][i-1]>25:
            pfvg_ret+=data['close'][i+20]-data['close'][i]
            pfvg+=1
            pret_per+=(data['close'][i+20]-data['close'][i])/data['close'][i]
            i+=h-1
        elif data['low'][i-1]-data['high'][i+1]>25:
            nfvg_ret+=data['close'][i]-data['close'][i+20]
            nfvg+=1
            nret_per+=(data['close'][i]-data['close'][i+20])/data['close'][i]
            i+=h-1
        i+=1
    print(f'horizon {h}: ',pfvg_ret/pfvg,nfvg_ret/nfvg,pret_per*100/pfvg,nret_per*100/nfvg,pfvg,nfvg)

horizon 1:  122.70849428745252 112.82528749886066 0.23016111923997895 0.1861481598328795 4774 4388
horizon 2:  117.30052814432987 102.34744412439485 0.21497409625252262 0.170416725928337 3986 3677
horizon 3:  113.49175710957292 88.04272656354836 0.20361254046238023 0.13763662468809676 3562 3303
horizon 4:  98.29137171764498 77.82484877634892 0.19416717734719832 0.11919554888109396 3236 3002
horizon 5:  96.69027594178188 79.50526444028033 0.19352484357650776 0.11560651626369524 2963 2711
horizon 6:  97.56947957749516 78.06108113863992 0.18977071732561362 0.12086228304917106 2715 2500
horizon 7:  110.3901351884859 89.17594455999989 0.21067594217157481 0.13493523202208407 2510 2335
horizon 8:  102.88148364580469 83.34657768548993 0.20205930091203275 0.12708609649820457 2336 2164
horizon 9:  103.8784698205692 79.30877158270748 0.2219812969855832 0.1119406348725387 2178 2024
horizon 10:  81.97945837113978 72.46026997961573 0.18897781901204141 0.11586786758922617 2079 1874
horizon 11:  98.33

In [360]:
def find_recent_swing_range(data):
    highs=[]
    lows=[]
    for i in range(5,len(data)-5):
        window=data.iloc[i-5:i+6]
        if data["high"].iloc[i]==window["high"].max():
            highs.append((i,data["high"].iloc[i]))
        if data["low"].iloc[i]==window["low"].min():
            lows.append((i,data["low"].iloc[i]))
    if not highs or not lows:
        return None,None
    recent_high=highs[-1][1]
    recent_low=lows[-1][1]
    return recent_high,recent_low

In [435]:
# Break of structure and Liquidity
pfvg_ret=nfvg_ret=pfvg=nfvg=pret_per=nret_per=0
pret=[]
nret=[]
for i in range(8,len(data)-8):
    bos=False
    break_liq=False
    if (data['low'][i+1]-data['high'][i-1])/data['close'][i]>0.0005:
        high=0
        j=2
        for j in range(2,8):
            if data['high'][i-j]>high:
                high=data['high'][i-j]
        if high<data['low'][i+1]:
            bos=True
        recent_high,recent_low=find_recent_swing_range(data[i-50:i-j])
        if recent_low is None:
            continue 
        for k in range(j,j+10):
            if data['low'][i-k]<recent_low:
                break_liq=True  
        if bos and break_liq:
            pfvg_ret+=data['close'][i+16]-data['close'][i]
            pfvg+=1
            pret_per+=(data['close'][i+16]-data['close'][i])/data['close'][i]
            pret.append(data['close'][i+16]-data['close'][i])
    elif (data['low'][i-1]-data['high'][i+1])/data['close'][i]>0.0005:
        bos=False
        break_liq=False 
        low=1e20
        j=2
        for j in range(2,8):
            if data['low'][i-j]<low:
                low=data['low'][i-j]
        if low>data['high'][i+1]:
            bos=True 
        recent_high,recent_low=find_recent_swing_range(data[i-50:i-j])
        if recent_high is None:
            continue
        for k in range(j,j+10):
            if recent_high<data['high'][i-k]:
                break_liq=True
        if bos and break_liq:
            nfvg_ret+=data['close'][i]-data['close'][i+16]
            nfvg+=1
            nret_per+=(data['close'][i]-data['close'][i+16])/data['close'][i]
            nret.append(data['close'][i]-data['close'][i+16])

In [436]:
pfvg_ret/pfvg,nfvg_ret/nfvg,pfvg,nfvg

(48.18353568166142, 183.11486970063504, 638, 630)

In [437]:
len(pret)

638

In [439]:
sum(1 for x in pret if x<0)

305

In [ ]:
def unmitigated(data):
    for i in range(8,len(data)-8):
        if data['low'][i+1]-data['high'][i-1]>25:
            mitigated=False 
            for j in range(i+2,i+8):
                if data['close'][j]<data['low'][i+1]:
                    mitigated=True
                    break
            if mitigated:
                return "pos"
        elif data['low'][i-1]-data['high'][i+1]>25:
            mitigated=False
            for j in range(i+2,i+8):
                if data['close'][j]>data['high'][i+1]:
                    mitigated=True
                    break
            if mitigated:
                return "neg"
    return "none"

In [ ]:
def reaction(data):
    for i in range(8,len(data)-8):
        if data['low'][i+1]-data['high'][i-1]>25:
            reaction=True  
            for j in range(i+2,i+8):
                if data['close'][j]<data['high'][i-1]:
                    reaction=False
            if reaction:
                return "pos"
        elif data['low'][i-1]-data['high'][i+1]>25:
            reaction=True 
            for j in range(i+2,i+8):
                if data['close'][j]>data['low'][i-1]:
                    reaction=False
            if reaction:
                return "neg"
    return "none"

In [ ]:
def sup_res(data):
    for i in range(7,len(data)-7):
        if data['low'][i+1]-data['high'][i-1]>25:
            support=False
            for j in range(2,5):
                if data['high'][i-1]<data['high'][i-j]<data['low'][i+1]:
                    support=True
                if data['high'][i-j]>data['low'][i+1]:
                    support=False
                    break
            if support:
                return "pos"
        elif data['low'][i-1]-data['high'][i+1]>25:
            resistance=False
            for j in range(2,7):
                if data['high'][i+1]<data['low'][i-j]<data['low'][i-1]:
                    resistance=True 
                if data['low'][i-j]<data['high'][i+1]:
                    resistance=False
                    break
            if resistance:
                return "neg"
    return "none"

In [ ]:
unmitigated(data[100:6000].reset_index(drop=True))

'neg'

In [ ]:
count=0
none=0
i=20
while i<len(data)-20:
    u=unmitigated(data[i:i+20].reset_index(drop=True))
    r=reaction(data[i:i+20].reset_index(drop=True))
    sr=sup_res(data[i:i+20].reset_index(drop=True))
    if u=="pos" or u=="neg":
        count+=1
    if r=="pos" or r=="neg":
        count+=1
    if sr=="pos" or sr=="neg":
        count+=1
    if u=="none":
        none+=1 
    if r=="none":
        none+=1 
    if sr=="none":
        none+=1  
    i+=20

In [269]:
count,none

(3650, 7880)